<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/languages/python/mini_projects/experiment_video_audio_extraction_ffmpeg_moviepy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install `moviepy`

We'll use the `moviepy` library for video and audio manipulation. Let's install it first.

In [ ]:
!pip install moviepy

### Video to Audio Extractor

This function will take a video file path, extract its audio, and save it as a separate audio file. It will also save the video without audio if specified. You will need to provide the path to your video file.

In [22]:
from moviepy.editor import VideoFileClip
import os

def extract_audio_and_video(video_path, output_audio_path=None, output_video_no_audio_path=None):
    """
    Extracts audio from a video file and can optionally save the video without audio.

    Args:
        video_path (str): The path to the input video file.
        output_audio_path (str, optional): The path to save the extracted audio file.
                                           If None, defaults to 'output_audio.mp3' in the same directory as the video.
        output_video_no_audio_path (str, optional): The path to save the video without audio.
                                                    If None, the video without audio will not be saved.
    """
    try:
        clip = VideoFileClip(video_path)

        # Extract and save audio
        if output_audio_path is None:
            base, _ = os.path.splitext(video_path)
            output_audio_path = f"{base}_audio.mp3"

        print(f"Extracting audio to: {output_audio_path}")
        clip.audio.write_audiofile(output_audio_path)
        print("Audio extraction complete.")

        # Save video without audio (if requested)
        if output_video_no_audio_path:
            print(f"Extracting video without audio to: {output_video_no_audio_path}")
            clip.write_videofile(output_video_no_audio_path, audio=False)
            print("Video without audio extraction complete.")

        clip.close()
        print(f"Successfully processed: {video_path}")

    except Exception as e:
        print(f"An error occurred: {e}")
        print("Please ensure the video file exists and is accessible. You might also need to install `ffmpeg` if you encounter errors related to video/audio codecs. `moviepy` uses `ffmpeg` in the backend.")


### How to Use

1.  **Upload your video**: Upload your video file to your Colab environment or specify its full path if it's already accessible.
2.  **Specify paths**: Update `your_video_file.mp4`, `extracted_audio.mp3`, and `video_no_audio.mp4` with your desired file names and paths.
3.  **Run the cell**: Execute the following cell to perform the extraction.

### Upload Your Video

Run the cell below to upload your video file. A file picker will appear. Please select your video file.

In [ ]:
from google.colab import files

uploaded = files.upload()

if uploaded:
    # Get the name of the first uploaded file
    uploaded_filename = list(uploaded.keys())[0]
    print(f"Uploaded file: {uploaded_filename}")
    video_file = uploaded_filename
else:
    print("No file uploaded. Please upload a video file to continue.")
    video_file = None # Ensure video_file is None if no file is uploaded

### Efficient Extraction for Long Videos with `ffmpeg`

For very long videos, using `ffmpeg` directly from the command line is often more efficient than going through `moviepy`'s Python wrapper. `ffmpeg` is pre-installed in Colab.

**Make sure you have successfully uploaded your video using the cell above and that the `video_file` variable contains the correct name of your uploaded file.**

In [ ]:
import re
import subprocess

video_duration_seconds = 0

if 'uploaded_filename' in globals() and uploaded_filename:
    print(f"Getting duration for: {uploaded_filename}")
    try:
        # Use ffprobe to get the exact duration
        cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", uploaded_filename]
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        video_duration_seconds = float(result.stdout.strip())
        print(f"Original video duration: {video_duration_seconds:.2f} seconds")
    except subprocess.CalledProcessError as e:
        print(f"Error getting video duration with ffprobe: {e.stderr}")
    except ValueError:
        print(f"Could not parse duration from ffprobe output: {result.stdout}")
else:
    print("No uploaded video found. Please upload a video first to determine its duration.")

In [ ]:
# Extract audio using ffmpeg
# The 'uploaded_filename' variable should hold the name of your uploaded video.
# '{output_audio_ffmpeg}' is the desired name for the extracted audio file.

output_audio_ffmpeg = None # Initialize the variable

if 'uploaded_filename' in globals() and uploaded_filename and video_duration_seconds > 0: # Check if a file was actually uploaded and duration obtained
    output_audio_ffmpeg = 'extracted_audio_ffmpeg.mp3' # A new name for ffmpeg output
    print(f"Extracting audio from '{uploaded_filename}' to '{output_audio_ffmpeg}' using ffmpeg...")
    # Use -t flag with the precise video_duration_seconds to ensure exact length
    !ffmpeg -i "{uploaded_filename}" -vn -t {video_duration_seconds} -y "{output_audio_ffmpeg}"
    print("Audio extraction complete via ffmpeg.")
elif not uploaded_filename:
    print("No video file uploaded. Please upload a video first by running cell `2851ff07`.")
else:
    print("Could not get video duration or uploaded file not found. Audio extraction skipped.")


In [ ]:
# Extract video without audio using ffmpeg
# '{output_video_only_ffmpeg}' is the desired name for the video without audio.

output_video_only_ffmpeg = None # Initialize the variable

if 'uploaded_filename' in globals() and uploaded_filename and video_duration_seconds > 0: # Check if a file was actually uploaded and duration obtained
    output_video_only_ffmpeg = 'video_no_audio_ffmpeg.mp4' # A new name for ffmpeg output
    print(f"Extracting video without audio from '{uploaded_filename}' to '{output_video_only_ffmpeg}' using ffmpeg...")
    # Use -ss 0 (start at 0) and -to (end time) for potentially more precise cutting with -c:v copy
    !ffmpeg -ss 0 -i "{uploaded_filename}" -to {video_duration_seconds} -an -c:v copy -y "{output_video_only_ffmpeg}"
    print("Video without audio extraction complete via ffmpeg.")
    # Check if output files were successfully created before suggesting to check for them
    if output_audio_ffmpeg and output_video_only_ffmpeg:
        print(f"\nCheck your Colab files for '{output_audio_ffmpeg}' and '{output_video_only_ffmpeg}' (if successfully created).")
    elif output_audio_ffmpeg:
        print(f"\nCheck your Colab files for '{output_audio_ffmpeg}' (if successfully created).")
    elif output_video_only_ffmpeg:
        print(f"\nCheck your Colab files for '{output_video_only_ffmpeg}' (if successfully created).")
else:
    print("No output files were created because the video file was not found, duration could not be determined, or an error occurred during extraction.")


### Preview and Download Extracted Files

Below are options to preview the `ffmpeg` extracted audio and video files directly in the notebook and to download them.

#### Preview Audio

In [ ]:
from IPython.display import Audio, display

# Check if the audio file was created before attempting to play
if 'output_audio_ffmpeg' in globals() and os.path.exists(output_audio_ffmpeg):
    print(f"Playing audio: {output_audio_ffmpeg}")
    display(Audio(output_audio_ffmpeg))
else:
    print(f"Audio file '{output_audio_ffmpeg}' not found. Please ensure ffmpeg extraction was successful.")

#### Preview Video (without audio)

In [ ]:
from IPython.display import Video

# Check if the video file was created before attempting to play
if 'output_video_only_ffmpeg' in globals() and os.path.exists(output_video_only_ffmpeg):
    print(f"Playing video without audio: {output_video_only_ffmpeg}")
    display(Video(output_video_only_ffmpeg, embed=True, width=640, height=360))
else:
    print(f"Video file '{output_video_only_ffmpeg}' not found. Please ensure ffmpeg extraction was successful.")

#### Download Files

In [30]:
from IPython.display import HTML, display
import os

print("Here are your extracted files. Click the links to download them:")

html_links = []

# Provide download link for audio file
if 'output_audio_ffmpeg' in globals() and os.path.exists(output_audio_ffmpeg):
    audio_link = f"<a href='{output_audio_ffmpeg}' download='{output_audio_ffmpeg}'>Download Audio: {output_audio_ffmpeg}</a>"
    html_links.append(audio_link)
else:
    html_links.append(f"<p>Audio file '{output_audio_ffmpeg}' not found.</p>")

# Provide download link for video without audio file
if 'output_video_only_ffmpeg' in globals() and os.path.exists(output_video_only_ffmpeg):
    video_link = f"<a href='{output_video_only_ffmpeg}' download='{output_video_only_ffmpeg}'>Download Video (no audio): {output_video_only_ffmpeg}</a>"
    html_links.append(video_link)
else:
    html_links.append(f"<p>Video file '{output_video_only_ffmpeg}' not found.</p>")

display(HTML("<br>".join(html_links)))
print("\nDownload options provided.")


Here are your extracted files. Click the links to download them:



Download options provided.
